# M08 Persona Vectors


## 先恢复一个 toy persona direction，再对照共享 artifact

先用成对 prompt embedding 恢复一个小型 persona direction，扫一遍干预强度，再与共享教学 artifact 对照。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import math
import matplotlib.pyplot as plt
import numpy as np

payload = json.loads((root / "artifacts" / "m08_persona_vectors.json").read_text())
traits = ["helpful", "cautious", "concise"]

neutral = np.array([
    [0.48, 0.44, 0.53],
    [0.51, 0.41, 0.49],
    [0.46, 0.47, 0.55],
    [0.5, 0.43, 0.5],
])
mentor = neutral + np.array([0.18, 0.06, -0.03])
reviewer = neutral + np.array([0.04, 0.22, 0.15])

mentor_direction = (mentor - neutral).mean(axis=0)
mentor_direction = mentor_direction / np.linalg.norm(mentor_direction)
base_prompt = np.array([0.5, 0.4, 0.52])
strengths = np.linspace(0.0, 1.4, 8)
toy_scores = []
for strength in strengths:
    steered = base_prompt + strength * mentor_direction
    toy_scores.append(np.clip(steered, 0.0, 1.0))
toy_scores = np.array(toy_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for index, trait in enumerate(traits):
    axes[0].plot(strengths, toy_scores[:, index], marker="o", label=trait)
axes[0].set_title("Toy persona-direction sweep")
axes[0].set_xlabel("steering strength")
axes[0].set_ylim(0.0, 1.0)
axes[0].legend()

artifact_persona = payload["personas"][1]
before = [artifact_persona["scores_before"][trait] for trait in traits]
after = [artifact_persona["scores_after"][trait] for trait in traits]
x = range(len(traits))
axes[1].bar([index - 0.16 for index in x], before, width=0.32, label="before", color="#999999")
axes[1].bar([index + 0.16 for index in x], after, width=0.32, label="after", color="#1f5f8b")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(traits, rotation=20)
axes[1].set_ylim(0, 1)
axes[1].set_title(artifact_persona["label_en"])
axes[1].legend()
plt.tight_layout()

def cosine(values_a, values_b):
    numerator = sum(a * b for a, b in zip(values_a, values_b))
    denom = math.sqrt(sum(a * a for a in values_a) * sum(b * b for b in values_b))
    return numerator / denom

reference = payload["personas"][0]
for persona in payload["personas"][1:]:
    score = cosine(reference["vector"], persona["vector"])
    print(f"cosine({reference['label_en']}, {persona['label_en']}) = {score:.3f}")

artifact_vector = np.array(artifact_persona["vector"], dtype=float)
print("cosine(toy mentor direction, artifact mentor vector) =", round(cosine(mentor_direction, artifact_vector[:3]), 3))
print("Best toy strength by helpful-minus-concise tradeoff:", round(float(strengths[np.argmax(toy_scores[:, 0] - toy_scores[:, 2])]), 2))


## 小结

当你先在本地恢复一个方向，再去对照更丰富的共享 artifact 时，persona vector 这件事才更像“可操作机制”，而不只是好看的图。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 在看 artifact 前，先选一个你最关心的 trait。
- 预测向量相似度是否应该和行为相似度对齐。
- 先写一条通过方向控制 character 的风险。


### 运行后

- 比较前后 trait 变化，并说哪个 trait 看起来最可控。
- 写出 persona control 可能漂移或失稳的一个原因。


### 最后交付这些产物

- 1 份短 memo，说明 persona vectors 让什么变得可读。
- 1 段风险说明。
- 1 个关于稳定性的后续问题。


## 验收题

- persona vectors 让哪些原本模糊的 character 属性变得可比较、可测量？
- 即使一个向量方向在当前 artifact 里有效，character control 仍然可能在哪些地方漂移或失稳？
- 如果你要评估 persona control 的稳定性，你会在不同 prompt、任务或时序上怎么测？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
